In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# -----------------------------------------------
# Malzeme fonksiyonları
# -----------------------------------------------

def steel_stress_strain(eps, Es=200000, fy=420):
    """
    Çelik için gerilme-şekil değiştirme ilişkisi.
    
    Parametreler:
    eps : Gerinim
        Şekil değiştirme (strain) değeri.
    Es : Elastisite modülü, MPa
        Çeliğin elastik modülü (default: 200000 MPa).
    fy : Akma gerilmesi, MPa
    """
    sigma = Es * eps

    # Gerilme fy ve -fy sınırları arası 

    sigma = Es * eps
    sigma = np.clip(sigma, -fy, fy)     

    return sigma

def comp_stress_strain(eps, fcd=20):
    """
    Beton için gerilme-şekil değiştirme ilişkisi.
    
    Parametreler:
    eps : Gerinim
        Şekil değiştirme (strain) değeri.
    fcd : Betonun tasarım basınç dayanımı, MPa
        Betonun tasarım basınç dayanımı (default: 20 MPa)
    
    Beton basınç gerinimini gerilmeye dönüştürür
    Basit sabit gerilme modeli kullanılmıştır.

    eps : Beton gerinimi
    fcd : Tasarım beton basınç dayanımı
    """

    sigma = np.zeros_like(eps)

    # Beton yalnızca basınçta gerilme taşır
    compression = eps < 0
    sigma[compression] = -fcd

    return sigma

#-----------------------------------------------
# Kesit ve malzeme bilgileri
# ----------------------------------------------

b = 300  # Kesit genişliği, mm
h= 500   # Kesit yüksekliği, mm
d = 450  # Etkin yükseklik, mm
c = 50   # Donatı konumu, mm

a=h

eps_cu = -0.0035 # Beton nihai basınç gerinimi
eps_su = 0.01    # Çelik gerinimi
n_step = 100 

#-----------------------------------------------
# Nötr eksen değerleri
#----------------------------------------------

yn_sup = 0
yn_inf = -d * eps_cu / (eps_su - eps_cu)

yn = np.linspace(yn_sup,yn_inf, n_step)

psi = eps_su / (d-yn)

eps_s1 = np.full(n_step, eps_su)
eps_s2 = -psi * (yn-c)

sig_s1 = steel_stress_strain(eps_s1)
sig_s2 = steel_stress_strain(eps_s2)

#-----------------------------------------------
# Beton kuvveti ve momenti
#-----------------------------------------------

Nc = np.zeros(n_step)
Mc = np.zeros(n_step)
ygc_values = np.zeros(n_step)

for i in range(n_step):
    
    y = np.linspace(0, yn[i], 100)

    eps_c = -(yn[i] - y) * psi[i]

    sig_c = comp_stress_strain(eps_c)

    # Sayısal entegrasyon
    Nc[i] = b * np.trapezoid(sig_c, y)

    denominator = np.trapezoid(sig_c * y, y)

    if abs(denominator) > 1e-12:
        ygc = np.trapezoid(sig_c * y, y) / denominator
    else:
        ygc = 0

    ygc_values[i] = ygc
    Mc[i] = Nc[i] * (a / 2 - ygc)

#-----------------------------------------------
# Toplam kuvvet ve moment
#-----------------------------------------------

Nrd_2 = Nc +As * sig_s1 + As *sig_s2

Mrd_2 = (-Mc + As * sig_s1 * (a / 2 - c) + As * sig_s2 * (a/2 - c))
 
#-----------------------------------------------
# Sonuçların yazdırılması
# ----------------------------------------------

print("Hesaplama tamamlandı.")
print(f"En büyük eksenel kuvvet:{np.max(Nrd_2):.2f}")
print(f"En küçük eksenel kuvvet:{np.min(Nrd_2):.2f}")
print(f"En büyük moment: {np.max(Mrd_2):.2f}")
print(f"En küçük moment: {np.min(Mrd_2):.2f}")

#------------------------------------------------
# N-M etkileşim grafiği
#------------------------------------------------

plt.figure(figsize=(8,6))

plt.plot(Nrd_2, Mrd_2, linewidth=2)

plt.xlabel("Eksenel kuvvet, N")
plt.ylabel("Moment, Nmm")
plt.title("N-M Etkileşim Diyagramı")
plt.grid(True)

plt.show()


NameError: name 'nstep' is not defined